# 第 8 课：CS336 桥梁 — 从 token 到自回归生成

**硬前提：你已经完成 PyTorch 速成课 1–7。** 本课不会复习 tensor、autograd、`nn.Module` 或训练循环。目标不是替你完成 CS336 Assignment 1，而是建立一条完整的数据流：

`文本 → byte/token IDs → embedding → causal attention → logits → next token`

学完后，你应该能读懂 Assignment 1 的接口与 shape，并知道 inference engine 到底执行什么。练习在 `practice/08_cs336_bridge.py`。

主要资料：[CS336 Spring 2026](https://stanford-cs336.github.io/spring2026/)、[Assignment 1](https://github.com/stanford-cs336/assignment1-basics)、[PyTorch SDPA](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html)。

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
print(torch.__version__)

2.12.1


## 1. 模型不读文字，只读整数

CS336 Assignment 1 从 byte-level BPE 开始，因为任意 Unicode 文本都能先编码为 UTF-8 bytes。byte vocabulary 只有 256 个基础值，不会出现 unknown byte。BPE 再把常见的相邻 byte 序列合并成较长 token，以缩短 sequence。

这里暂时不实现 BPE；先用 UTF-8 byte IDs 看清数据类型。正式 BPE 留给 CS336 作业独立完成。

In [2]:
text = "牛A"
raw = text.encode("utf-8")
token_ids = torch.tensor(list(raw), dtype=torch.long)

print("characters:", len(text), "bytes:", len(raw))
print("byte IDs:", token_ids.tolist())
assert raw.decode("utf-8") == text
assert token_ids.dtype == torch.long

characters: 2 bytes: 4
byte IDs: [231, 137, 155, 65]


## 2. Language modeling 只是错开一格

给定 token 序列 `[t0, t1, t2, t3]`：

- 输入 `x = [t0, t1, t2]`
- 标签 `y = [t1, t2, t3]`

模型在每个位置输出 vocabulary 上的 logits。训练时所有位置并行预测；推理时只取最后一个位置，生成一个 token 后再继续。

In [3]:
tokens = torch.tensor([10, 20, 30, 40, 50])
x, y = tokens[:-1], tokens[1:]
print("x:", x.tolist())
print("y:", y.tolist())
assert torch.equal(x[1:], y[:-1])

x: [10, 20, 30, 40]
y: [20, 30, 40, 50]


## 3. CS336 的核心习惯：每段代码都做 resource accounting

常用符号：`B`=batch，`T`=sequence length，`D`=model width，`H`=heads，`Dh=D/H`。

知道 shape 只是起点。还要能回答：参数有多少？activation 占多少 bytes？一次 matmul 有多少 FLOPs？

一个现代 decoder block 的粗略参数账本：attention 的 Q/K/V/O 是 `4D²`；SwiGLU 有三个矩阵，是 `3D×Dff`；两个 RMSNorm 约 `2D`。token embedding 若与 LM head tied，只算一次 `V×D`。

In [4]:
V, L, D, Dff = 32_000, 12, 768, 2048
embedding = V * D
per_layer = 4 * D * D + 3 * D * Dff + 2 * D
parameters = embedding + L * per_layer
weight_bytes_bf16 = parameters * 2

print(f"parameters: {parameters / 1e6:.1f}M")
print(f"BF16 weights: {weight_bytes_bf16 / 2**20:.1f} MiB")
assert parameters == embedding + L * per_layer

parameters: 109.5M
BF16 weights: 208.9 MiB


## 4. Causal attention：位置 t 看不到未来

Attention 的语义是 `softmax(QKᵀ / √Dh) V`。decoder-only LM 还要加 causal mask：第 `t` 个位置只能读取 `0..t`。

这里调用 PyTorch SDPA 作为正确性 oracle。它让你观察语义，但 **CS336 Assignment 1 要求你从零实现，不能把这个 built-in 当答案**。最重要的不变量：修改未来 token 的 K/V，不应改变更早位置的输出。

In [5]:
q = torch.ones(1, 1, 3, 1)
k = torch.ones(1, 1, 3, 1)
v = torch.tensor([[[[2.0], [4.0], [9.0]]]])
out = F.scaled_dot_product_attention(q, k, v, is_causal=True)

v_changed = v.clone()
v_changed[:, :, 2] = 999
out_changed = F.scaled_dot_product_attention(q, k, v_changed, is_causal=True)

print(out.flatten().tolist())
assert torch.allclose(out, torch.tensor([[[[2.0], [3.0], [5.0]]]]))
assert torch.allclose(out[:, :, :2], out_changed[:, :, :2])

[2.0, 3.0, 5.0]


## 5. 从 vanilla Transformer 到 CS336 的现代 recipe

不要把 `nn.TransformerEncoderLayer` 当成现代 LLM 架构。CS336 Assignment 1 要你理解并实现：**pre-norm RMSNorm、SwiGLU、RoPE、causal multi-head attention**。推理阶段还要继续追问 MHA/MQA/GQA 如何改变 KV cache。

本课用 built-in layer 只是语义基线：先验证 causal behavior 与 logits shape。正式作业再把每个部件换成自己的实现。

## 6. 用 built-ins 组装语义基线

下面的 tiny model 只用于连接数据流：embedding 把 IDs 变成向量，Transformer block 混合上下文，LM head 把每个位置投影回 vocabulary logits。

它不是 CS336 作业答案：Assignment 1 会要求你自己实现 linear、embedding、RMSNorm、SwiGLU、RoPE、attention 和 Transformer LM。

In [6]:
class TinyBuiltinLM(nn.Module):
    def __init__(self, vocab_size=256, context_length=32, d_model=32, num_heads=4):
        super().__init__()
        self.context_length = context_length
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(context_length, d_model)
        self.block = nn.TransformerEncoderLayer(
            d_model, num_heads, dim_feedforward=4 * d_model,
            dropout=0.0, batch_first=True, norm_first=True,
        )
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, token_ids):
        _, sequence_length = token_ids.shape
        if sequence_length > self.context_length:
            raise ValueError("sequence exceeds context length")
        positions = torch.arange(sequence_length, device=token_ids.device)
        x = self.token_embedding(token_ids) + self.position_embedding(positions)
        causal_mask = torch.triu(
            torch.ones(sequence_length, sequence_length, dtype=torch.bool, device=token_ids.device),
            diagonal=1,
        )
        return self.lm_head(self.block(x, src_mask=causal_mask))

model = TinyBuiltinLM().eval()
sample = torch.tensor([[72, 105, 33]])
logits = model(sample)
assert logits.shape == (1, 3, 256)
print("token IDs:", sample.shape, "-> logits:", logits.shape)

token IDs: torch.Size([1, 3]) -> logits: torch.Size([1, 3, 256])


## 7. Training 与 inference 在哪里分叉

训练：一次 forward 得到 `(B,T,V)`，与右移一位的 targets 算 cross entropy。

推理：prompt 的第一次完整 forward 叫 **prefill**；之后每轮只生成一个 token，叫 **decode**。下面的朴素版本每轮重算全部历史，没有 KV cache，复杂但正确性容易验证。inference engine 的第一项核心优化，就是缓存每层过去的 K/V。

MHA KV cache bytes：`2 × layers × batch × heads × sequence × head_dim × element_bytes`。开头的 2 是 K 和 V。

In [7]:
@torch.no_grad()
def generate_greedy(model, token_ids, max_new_tokens):
    for _ in range(max_new_tokens):
        window = token_ids[:, -model.context_length:]
        next_token = model(window)[:, -1].argmax(dim=-1, keepdim=True)
        token_ids = torch.cat((token_ids, next_token), dim=1)
    return token_ids

generated = generate_greedy(model, sample, max_new_tokens=4)
print(generated.tolist())
assert generated.shape == (1, 7)

[[72, 105, 33, 136, 214, 17, 26]]


## 8. 练习与最终 Assignment

运行：

```bash
uv run pytest practice/08_cs336_bridge.py -v
```

前 8 题依次检查 UTF-8 IDs、next-token pairs、batch、Transformer 参数账本、causal SDPA、LM loss、top-k filtering 与 MHA/GQA KV-cache memory。

**第 9 题是综合 Assignment**：只用 PyTorch built-ins 组装 `TinyBuiltinLM`，通过 output shape 与 causal invariance 测试。完成它以后，再进入正式 CS336 Assignment 1，从零实现相同语义。

有概念或测试不清楚时，可以把你的理解和实际输出交给 agent 检查；正式 CS336 作业只接受讲解、提示和 code review，不让 agent 代写实现。